# 06 · Weather Sensitivity Analysis
**Team BUTA · DS-GA 1007**

Neighboring areas get the same weather. Do their 311 complaints react to it the same way?

This notebook answers four questions:
1. **Which complaint types follow the weather?** (citywide)
2. **How big is the effect?** We use a Poisson model with weekday, month and year controls, so a cold snap is compared with a normal day in the same month.
3. **Does sensitivity differ between ZIP codes, and are neighboring ZIPs alike?** This tests the "100-meter inequality" idea at ZIP scale.
4. **Do older buildings generate more heat complaints per apartment?**

Inputs: the tables built in **05_build_project.ipynb**. Every figure is saved to `reports/figures/`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config, analysis as A, plots as P

pd.set_option("display.float_format", "{:,.2f}".format)
panel = pd.read_parquet(config.PANEL_ZIP_DAILY_FILE)
pluto = pd.read_parquet(config.CLEAN_PLUTO_FILE)
buildings = pd.read_parquet(config.BUILDING_HEAT_FILE)
print(f"ZIP-day panel: {len(panel):,} rows | ZIPs: {panel.incident_zip.nunique()} | days: {panel.date.nunique()}")

## 1. Which complaints follow the weather?

In [ ]:
fig = P.weather_correlation_heatmap(panel); P.save(fig, "01_weather_correlations")

**Reading it:**
- Heat/hot-water complaints move almost one-for-one with cold (strong negative correlation with temperature).
- Outdoor issues (street noise, rodents, dirty conditions) rise with warmth.
- Sewer complaints follow rain.
- **Residential noise does not respond to weather at all.** It happens indoors, so it acts as a natural "control" category.

## 2. How big is the effect? (citywide)

In [ ]:
heat_daily = A.citywide_daily(panel, A.HEAT_SPEC)
sewer_daily = A.citywide_daily(panel, A.SEWER_SPEC)
P.save(P.heat_vs_temperature(heat_daily), "02_heat_vs_temperature")
P.save(P.sewer_vs_rain(sewer_daily), "03_sewer_vs_rain")

city = pd.DataFrame([A.citywide_summary(panel, s) for s in (A.HEAT_SPEC, A.SEWER_SPEC)])
city[["complaint_type", "effect_label", "pct_effect", "ci_low", "ci_high",
      "pseudo_r2_controls_only", "pseudo_r2_with_weather"]]

**Model:** `complaints ~ weather (today + yesterday) + weekday + month + year`. This is a quasi-Poisson GLM, so the standard errors allow for overdispersion.

- Because month is controlled for, the weather effect is the **extra** complaints from a day that is colder or wetter than usual *for that month*.
- Year dummies absorb the steady growth in 311 usage.
- **Pseudo-R²** is the share of deviance explained. Comparing "controls only" with "with weather" shows how much the weather itself adds beyond the calendar.

## 3. Does sensitivity differ by ZIP code?
We fit the same model separately for each ZIP with enough complaints. Each ZIP gets its own sensitivity and confidence interval.

In [ ]:
centroids = A.zip_centroids(pluto)
heat_zip = A.zip_sensitivities(panel, A.HEAT_SPEC, min_complaints=300)
sewer_zip = A.zip_sensitivities(panel, A.SEWER_SPEC, min_complaints=150)
print(f"Heat: {len(heat_zip)} ZIPs | Sewer: {len(sewer_zip)} ZIPs")
pd.concat({"Heat (% per 10°F colder)": heat_zip.pct_effect.describe(),
           "Sewer (% per +1 in rain)": sewer_zip.pct_effect.describe()}, axis=1)

In [ ]:
P.save(P.sensitivity_map(heat_zip, centroids, "Heat-complaint sensitivity to cold, by ZIP",
                         "% more complaints per 10°F colder"), "04_heat_sensitivity_map")
P.save(P.sensitivity_ranking(heat_zip, 10, "Most / least cold-sensitive ZIPs (95% CI)",
                             "% more heat complaints per 10°F colder"), "05_heat_sensitivity_ranking")
P.save(P.sensitivity_map(sewer_zip, centroids, "Sewer-complaint sensitivity to rain, by ZIP",
                         "% more complaints per +1 inch rain"), "06_sewer_sensitivity_map")

### 3b. Are neighboring ZIPs alike? (permutation test)

For every ZIP we take its **nearest neighbor** and measure the gap in sensitivity. We then shuffle the sensitivities across ZIPs 2,000 times to see what gap random pairs would produce.

- **Actual gap clearly smaller than random:** sensitivity is spatially clustered. Neighbors behave alike, which suggests an area-level cause such as a drainage system.
- **Actual gap about the same as random:** neighbors are no more alike than ZIPs across the city. Differences are hyperlocal, which supports the 100-meter-inequality idea.

In [ ]:
tests = {"Heat vs cold": A.neighbor_gap_test(heat_zip, centroids),
         "Sewer vs rain": A.neighbor_gap_test(sewer_zip, centroids)}
P.save(P.neighbor_test_plot(tests), "07_neighbor_test")
pd.DataFrame({k: {m: v[m] for m in ("observed_mean_gap", "random_mean_gap", "p_value", "median_neighbor_km")}
              for k, v in tests.items()})

In [ ]:
# The most different neighboring pairs: candidates for a closer look
tests["Heat vs cold"]["pairs"].head(6)

In [ ]:
# Does the housing stock explain a ZIP's cold sensitivity?
heat_zip[["pct_effect", "share_units_pre1940", "median_year_built", "rate_per_10k_units_per_day"]].corr()["pct_effect"]

## 4. Building level: does age matter?

In [ ]:
A.era_summary(buildings)

In [ ]:
rr = A.building_rate_model(buildings)
P.save(P.building_era_plot(rr), "08_building_era")
print("Building-seasons in model:", rr.attrs["n_obs"])
rr

**Model:** heat complaints per building-season ~ era + borough + size band + season, with `log(apartments)` as the offset. The ratios are therefore **complaints per apartment** relative to buildings from 2000 onward, with like compared to like on borough and size.

## 5. Takeaways and caveats
Fill in the numbers from the outputs above when writing the slides.

- **Weather is a major driver of some complaint types.** Cold explains a large share of heat complaints even after removing seasonality, and heavy rain multiplies sewer complaints.
- **Sensitivity is uneven across ZIPs.** Heat sensitivity varies **hyperlocally**: neighbors are no more alike than random ZIPs. Rain sensitivity is **spatially clustered**, which is consistent with shared drainage infrastructure.
- **Building age matters for the level of heat complaints.** Pre-1940 buildings generate several times more complaints per apartment than modern ones.

**Caveats**
- 311 records **reports**, not conditions. Areas that report less are not necessarily better off.
- Weather comes from a single Central Park point (Open-Meteo reanalysis). A NOAA station is the official alternative.
- ZIPs are coarse. The next step toward "100 m" is to repeat section 3 on a 200 m grid or at building level.
- The ZIP map uses unit-weighted centroids from PLUTO instead of ZIP polygons.